# Evon v3.3 Q4_K_M on Kaggle 2× T4 with llama-server: speed and Telugu test

Loads `anilkumarlucky/gnani-evon-v3.3-30b-a3b-GGUF` (Q4_K_M) into `llama-server`, split across both T4s, and answers three questions for Kavach:

1. **Does it fit?** Full GPU offload first; if a T4 runs out of memory, the first N layers' MoE experts move to CPU RAM (`--n-cpu-moe`) and we retry.
2. **How fast?** Prompt (prefill) and generation tokens/s, one request and two at once (Prosecutor + Defender run in parallel).
3. **Is the Telugu usable?** The five Kavach roles on one synthetic scam call and one genuine bank call (false-positive check), with thinking off and on.

**Before Run all**
1. Settings → Accelerator: **GPU T4 ×2**. Settings → Internet: **on**.
2. Add-ons → Secrets → add `HF` = your Hugging Face token, and tick it for this notebook.

Everything is written to `/kaggle/working/evon_t4_report.md` (+ `.json`) at the end; download that and bring it back.

In [ ]:
# 1. Settings
import os, json, time, subprocess, shutil, re, threading
from pathlib import Path

GGUF_REPO  = "anilkumarlucky/gnani-evon-v3.3-30b-a3b-GGUF"
GGUF_FILE  = "gnani-evon-v3.3-30b-a3b-Q4_K_M.gguf"
BASE_REPO  = "gnani/gnani-evon-v3.3-30b-a3b"      # chat_template.jinja lives here, not in tokenizer_config
LLAMA_SHA  = "b3fa699"                             # the llama.cpp commit the GGUF was converted with
SRC        = Path("/tmp/llama.cpp")
BIN        = Path("/kaggle/working/llama-bin")     # keep binaries so a rerun in the same session skips the build
MODEL_DIR  = Path("/tmp/evon")                     # /kaggle/working is capped at 20 GB, so the model goes to /tmp
PORT       = 8080
URL        = f"http://127.0.0.1:{PORT}"
CTX, SLOTS = 16384, 2                              # 8k tokens per slot, two slots for parallel roles
CPU_MOE_STEPS = [0, 6, 12, 20]                     # retry ladder if the T4s run out of memory

REPORT = {"started": time.strftime("%Y-%m-%d %H:%M:%S")}
def note(k, v):
    REPORT[k] = v
    print(f"{k}: {v}")

In [ ]:
# 2. Hardware check
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv
!nvcc --version | tail -2
!free -g | head -2; df -h /tmp /kaggle/working | tail -2
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.split("\n")
gpus = [g for g in gpus if g.strip()]
note("gpus", gpus)
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"

## Build llama-server with CUDA (about 15–25 min, once per session)

Only the T4's architecture (sm_75) is compiled, which cuts the build time a lot. Static linking keeps the binary self-contained. Kaggle ships no CUDA stub library, so CMake is pointed at the real driver in /usr/local/nvidia/lib64.

In [ ]:
# 3. Build
SERVER = BIN / "llama-server"
if not SERVER.exists():
    t = time.time()
    if not SRC.exists():
        !git clone -q --filter=blob:none https://github.com/ggml-org/llama.cpp {SRC}
    !git -C {SRC} checkout -q {LLAMA_SHA}
    !cd {SRC} && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_cuda_driver_LIBRARY=/usr/local/nvidia/lib64/libcuda.so > /tmp/cmake.log 2>&1 || tail -30 /tmp/cmake.log
    !cd {SRC} && cmake --build build --target llama-server -j {os.cpu_count()} > /tmp/build.log 2>&1 || tail -40 /tmp/build.log
    BIN.mkdir(parents=True, exist_ok=True)
    shutil.copy(SRC / "build/bin/llama-server", SERVER)
    note("build_minutes", round((time.time() - t) / 60, 1))
assert SERVER.exists(), "build failed, see the log tail above"
note("llama_cpp", LLAMA_SHA)

In [ ]:
# 4. Download the Q4_K_M GGUF (private repo) and Evon's chat template
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF")
t = time.time()
MODEL = Path(hf_hub_download(GGUF_REPO, GGUF_FILE, local_dir=MODEL_DIR))
TEMPLATE = Path(hf_hub_download(BASE_REPO, "chat_template.jinja", local_dir=MODEL_DIR))
note("model_gb", round(MODEL.stat().st_size / 1e9, 2))
note("download_minutes", round((time.time() - t) / 60, 1))

## Start the server

`-sm layer -ts 1,1` puts half the layers on each T4. If loading fails with an out-of-memory error, the next step of `CPU_MOE_STEPS` keeps that many layers' experts in CPU RAM (slower, but only 6 of 128 experts are active per token).

In [ ]:
# 5. Start llama-server, stepping down the GPU load until it fits
import requests

def gpu_mem():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().splitlines()
    return [f"{int(u)/1024:.1f}/{int(t)/1024:.1f} GB" for u, t in (l.split(", ") for l in out)]

def start_server(n_cpu_moe):
    log = open(f"/tmp/server_cpumoe{n_cpu_moe}.log", "w")
    cmd = [str(SERVER), "-m", str(MODEL), "--chat-template-file", str(TEMPLATE),
           "-ngl", "999", "-sm", "layer", "-ts", "1,1", "-c", str(CTX), "-np", str(SLOTS),
           "-fa", "auto", "--host", "127.0.0.1", "--port", str(PORT), "--metrics"]
    if n_cpu_moe:
        cmd += ["--n-cpu-moe", str(n_cpu_moe)]
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    t = time.time()
    while time.time() - t < 900:
        if proc.poll() is not None:
            return None, log.name
        try:
            if requests.get(f"{URL}/health", timeout=2).status_code == 200:
                return proc, log.name
        except requests.RequestException:
            pass
        time.sleep(5)
    proc.kill()
    return None, log.name

server = None
for n in CPU_MOE_STEPS:
    t = time.time()
    server, logname = start_server(n)
    if server:
        note("n_cpu_moe", n)
        note("load_seconds", round(time.time() - t))
        note("vram_after_load", gpu_mem())
        break
    tail = open(logname).read()[-1500:]
    print(f"--- n_cpu_moe={n} failed; log tail:\n{tail}")
    if not re.search(r"out of memory|failed to allocate|cudaMalloc", tail, re.I):
        raise SystemExit("Server failed for a reason other than memory; see the log tail above.")
assert server, "Did not fit even with the largest --n-cpu-moe step"

In [ ]:
# 6. Client helper. Thinking is on by default in Evon's template, so every call says which it wants.
def chat(messages, max_tokens=512, thinking=False, schema=None, temperature=0.3, cache_prompt=True):
    body = {"messages": messages, "max_tokens": max_tokens, "temperature": temperature,
            "chat_template_kwargs": {"enable_thinking": thinking}, "cache_prompt": cache_prompt}
    if schema:
        body["response_format"] = {"type": "json_object", "schema": schema}
    t = time.time()
    r = requests.post(f"{URL}/v1/chat/completions", json=body, timeout=900)
    r.raise_for_status()
    j = r.json()
    msg = j["choices"][0]["message"]
    tm = j.get("timings", {})
    return {"content": msg.get("content") or "", "reasoning": msg.get("reasoning_content") or "",
            "wall_s": round(time.time() - t, 2),
            "prompt_tokens": tm.get("prompt_n"), "prompt_tps": round(tm.get("prompt_per_second", 0), 1),
            "gen_tokens": tm.get("predicted_n"), "gen_tps": round(tm.get("predicted_per_second", 0), 1),
            "finish": j["choices"][0].get("finish_reason")}

def n_tokens(text):
    return len(requests.post(f"{URL}/tokenize", json={"content": text}).json()["tokens"])

warm = chat([{"role": "user", "content": "Say OK."}], max_tokens=8)
print(warm)

## Speed

In [ ]:
# 7. Generation and prefill speed, then two requests at once
from concurrent.futures import ThreadPoolExecutor

speed = {}
r = chat([{"role": "user", "content": "Write 200 words about why people should never share an OTP on a phone call."}],
         max_tokens=300, temperature=0.7, cache_prompt=False)
speed["single_gen"] = {k: r[k] for k in ("prompt_tokens", "prompt_tps", "gen_tokens", "gen_tps", "wall_s")}

long_text = ("A bank customer receives a call. The caller claims to be from the cyber police and says a parcel "
             "in the customer's name contained drugs. ") * 120
r = chat([{"role": "user", "content": long_text + "\nSummarise the above in one sentence."}],
         max_tokens=40, cache_prompt=False)
speed["prefill_long"] = {k: r[k] for k in ("prompt_tokens", "prompt_tps", "gen_tokens", "gen_tps", "wall_s")}

prompts = ["List five signs of a phone scam.", "List five signs that a bank call is genuine."]
t = time.time()
with ThreadPoolExecutor(2) as ex:
    rs = list(ex.map(lambda p: chat([{"role": "user", "content": p}], max_tokens=200, temperature=0.7,
                                    cache_prompt=False), prompts))
speed["two_parallel"] = {"wall_s": round(time.time() - t, 2), "per_request_gen_tps": [x["gen_tps"] for x in rs],
                         "total_gen_tokens": sum(x["gen_tokens"] for x in rs)}
speed["two_parallel"]["aggregate_gen_tps"] = round(speed["two_parallel"]["total_gen_tokens"] / speed["two_parallel"]["wall_s"], 1)

te = "మీ ఆధార్ నంబర్‌తో ఒక పార్సెల్‌లో డ్రగ్స్ దొరికాయి. ఈ విషయం ఎవరికీ చెప్పకూడదు, వెంటనే వీడియో కాల్‌లో ఉండండి."
speed["telugu_tokens_per_word"] = round(n_tokens(te) / len(te.split()), 2)
note("speed", speed)
note("vram_after_speed", gpu_mem())

## Kavach roles on two synthetic calls

Both transcripts are short, hand-written test calls in the Telugu–English mix Prisma is expected to return. The first is a "digital arrest" scam; the second is a genuine card-delivery call that must **not** be flagged.

In [ ]:
# 8. Test calls and role prompts
CALLS = {
"scam_digital_arrest": '''Caller: హలో, నేను Mumbai Cyber Crime police station నుంచి Inspector Sharma మాట్లాడుతున్నాను.
Caller: మీ ఆధార్ నంబర్‌తో ఒక parcel లో drugs దొరికాయి. మీ మీద FIR register అయింది.
User: అయ్యో, నాకు ఏమీ తెలియదు సార్.
Caller: ఈ case చాలా serious. మీరు ఇప్పుడే video call లో ఉండాలి, ఈ విషయం ఎవరికీ చెప్పకూడదు, లేకపోతే arrest warrant issue చేస్తాం.
Caller: మీ bank account verify చేయాలి. మీ savings మొత్తం RBI safe account కి transfer చేయండి, investigation తర్వాత return చేస్తాం.''',
"genuine_card_delivery": '''Caller: నమస్కారం, నేను SBI Card customer care నుంచి మాట్లాడుతున్నాను.
Caller: మీ కొత్త credit card రేపు courier లో వస్తుంది. Delivery సమయంలో ఏదైనా ID proof చూపించండి.
User: సరే, ఏమైనా OTP చెప్పాలా?
Caller: లేదు సార్, మేము ఎప్పుడూ OTP కానీ PIN కానీ అడగము. ఏదైనా సందేహం ఉంటే card వెనుక ఉన్న number కి call చేయండి.
User: సరే, thank you.'''}

SYS = "You are part of Kavach, a second listener that protects elderly Telugu speakers from phone scams. Be precise and quote the transcript."

def prosecutor(call):
    return [{"role": "system", "content": SYS},
            {"role": "user", "content": f"Transcript:\n{call}\n\nAs the PROSECUTOR, argue that this is a scam. "
             "List each scam tactic you see (e.g. impersonating police, secrecy demand, arrest threat, money transfer to a 'safe account'), "
             "with the exact quote. If you find no real evidence, say so honestly. Answer in English, at most 120 words."}]

def defender(call):
    return [{"role": "system", "content": SYS},
            {"role": "user", "content": f"Transcript:\n{call}\n\nAs the DEFENDER, argue that this call could be genuine. "
             "Give the strongest innocent explanation for each suspicious line, and say plainly where none exists. Answer in English, at most 120 words."}]

JUDGE_SCHEMA = {"type": "object", "properties": {
    "verdict": {"type": "string", "enum": ["SCAM", "SUSPICIOUS", "SAFE"]},
    "confidence": {"type": "number"},
    "tactics": {"type": "array", "items": {"type": "string"}},
    "reason": {"type": "string"}},
    "required": ["verdict", "confidence", "tactics", "reason"]}

def judge(call, pro, dfn):
    return [{"role": "system", "content": SYS},
            {"role": "user", "content": f"Transcript:\n{call}\n\nProsecutor:\n{pro}\n\nDefender:\n{dfn}\n\n"
             "As the JUDGE, weigh both sides and return JSON: verdict (SCAM, SUSPICIOUS or SAFE), confidence 0-1, "
             "tactics (short English labels, empty if none), reason (one English sentence)."}]

def warning(call, verdict):
    return [{"role": "system", "content": SYS},
            {"role": "user", "content": f"Transcript:\n{call}\n\nVerdict: {verdict}\n\n"
             "Write the message Kavach will speak aloud to the elderly listener, in simple spoken Telugu (Telugu script), "
             "at most two short sentences. If the verdict is SCAM or SUSPICIOUS, tell them to hang up and not send money or share OTP. "
             "If SAFE, reassure them briefly. Output only the Telugu message."}]

def reporter(call, verdict):
    return [{"role": "system", "content": SYS},
            {"role": "user", "content": f"Transcript:\n{call}\n\nVerdict: {verdict}\n\n"
             "As the REPORTER, draft a short complaint the family can file on the 1930 helpline / cybercrime.gov.in: "
             "what was claimed, what was demanded, and the tactics used. English, at most 100 words, then the same summary in Telugu."}]

def telugu_ratio(s):
    letters = [c for c in s if c.isalpha()]
    return round(sum('ఀ' <= c <= '౿' for c in letters) / max(len(letters), 1), 2)

In [ ]:
# 9. Run the pipeline on both calls, thinking off (the fast path) and on (to see if it changes verdicts)
results = {}
for think in (False, True):
    for name, call in CALLS.items():
        key = f"{name}|thinking={think}"
        t = time.time()
        with ThreadPoolExecutor(2) as ex:   # Prosecutor and Defender in parallel, as Kavach will run them
            pro, dfn = ex.map(lambda m: chat(m, max_tokens=1500 if think else 300, thinking=think),
                              [prosecutor(call), defender(call)])
        # the Judge is grammar-constrained JSON, so it always runs with thinking off
        jd = chat(judge(call, pro["content"], dfn["content"]), max_tokens=300, schema=JUDGE_SCHEMA, temperature=0.1)
        try:
            verdict = json.loads(jd["content"])
        except json.JSONDecodeError:
            verdict = {"verdict": "PARSE_ERROR", "raw": jd["content"]}
        wn = chat(warning(call, verdict.get("verdict")), max_tokens=1000 if think else 150, thinking=think)
        rp = chat(reporter(call, verdict.get("verdict")), max_tokens=1500 if think else 400, thinking=think)
        results[key] = {
            "verdict": verdict,
            "seconds_to_verdict": round(max(pro["wall_s"], dfn["wall_s"]) + jd["wall_s"], 1),
            "seconds_total": round(time.time() - t, 1),
            "prosecutor": pro["content"], "defender": dfn["content"],
            "warning_te": wn["content"], "warning_telugu_ratio": telugu_ratio(wn["content"]),
            "report": rp["content"],
            "gen_tps": [pro["gen_tps"], dfn["gen_tps"], jd["gen_tps"], wn["gen_tps"], rp["gen_tps"]],
            "finish": [pro["finish"], dfn["finish"], jd["finish"], wn["finish"], rp["finish"]]}
        print(f"\n===== {key}: {verdict.get('verdict')} ({results[key]['seconds_to_verdict']} s to verdict)")
        print("WARNING (te):", wn["content"])
note("vram_after_roles", gpu_mem())

In [ ]:
# 10. Automatic checks (Telugu quality itself still needs a human read: see the report)
checks = {}
for think in (False, True):
    s = results[f"scam_digital_arrest|thinking={think}"]
    g = results[f"genuine_card_delivery|thinking={think}"]
    checks[f"thinking={think}"] = {
        "scam_flagged": s["verdict"].get("verdict") in ("SCAM", "SUSPICIOUS"),
        "genuine_not_flagged": g["verdict"].get("verdict") == "SAFE",
        "warnings_mostly_telugu": min(s["warning_telugu_ratio"], g["warning_telugu_ratio"]) >= 0.8,
        "no_truncation": all(f == "stop" for f in s["finish"] + g["finish"]),
        "seconds_to_verdict_scam": s["seconds_to_verdict"]}
note("checks", checks)

In [ ]:
# 11. Write the report and stop the server
REPORT["results"] = results
REPORT["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
Path("/kaggle/working/evon_t4_report.json").write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8")

sp = REPORT["speed"]
lines = [f"# Evon v3.3 Q4_K_M on Kaggle 2x T4 ({REPORT['finished']})", "",
         f"- llama.cpp {REPORT['llama_cpp']}, model {REPORT['model_gb']} GB, --n-cpu-moe {REPORT['n_cpu_moe']}, load {REPORT['load_seconds']} s",
         f"- VRAM after load: {REPORT['vram_after_load']}; after roles: {REPORT['vram_after_roles']}",
         f"- Generation, one request: {sp['single_gen']['gen_tps']} tok/s; prefill on {sp['prefill_long']['prompt_tokens']} tokens: {sp['prefill_long']['prompt_tps']} tok/s",
         f"- Two requests at once: {sp['two_parallel']['per_request_gen_tps']} tok/s each, {sp['two_parallel']['aggregate_gen_tps']} tok/s combined",
         f"- Telugu tokens per word: {sp['telugu_tokens_per_word']}", "", "## Checks", "",
         "```", json.dumps(REPORT["checks"], indent=2), "```"]
for key, r in results.items():
    lines += ["", f"## {key}", "",
              f"**Verdict:** `{json.dumps(r['verdict'], ensure_ascii=False)}`, {r['seconds_to_verdict']} s to verdict, {r['seconds_total']} s for all five roles, gen tok/s {r['gen_tps']}",
              "", f"**Warning (Telugu, {r['warning_telugu_ratio']:.0%} Telugu script):** {r['warning_te']}",
              "", "**Prosecutor:**", "", r["prosecutor"], "", "**Defender:**", "", r["defender"],
              "", "**Report:**", "", r["report"]]
Path("/kaggle/working/evon_t4_report.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:12]))

server.terminate()
print("\nwrote /kaggle/working/evon_t4_report.md and .json")